# Whisper Small Transcription on Google Colab

This notebook demonstrates a clean, reproducible transcription workflow built around **Hugging Face Transformers** and **`openai/whisper-small`**.

### Design Decisions

- **Model loading strategy:** use `AutoModelForSpeechSeq2Seq` plus `AutoProcessor` for explicit control over precision, memory usage, and the final inference pipeline.
- **Inference strategy:** use the Transformers `automatic-speech-recognition` pipeline with `chunk_length_s=30`, which is the Hugging Face-recommended path for longer audio.
- **GPU optimization:** prefer **FP16** whenever CUDA is available, which is a good fit for a Colab **T4 GPU**.
- **Reproducibility:** keep dependencies minimal, expose configuration in one place, and save outputs to disk in both `.txt` and `.json` formats.

Run the notebook top to bottom. If you want the best performance, switch Colab to **`Runtime > Change runtime type > T4 GPU`** before running the cells.

## 1. Environment Setup

This section checks the runtime, reports whether CUDA is available, and confirms whether Colab is currently attached to a GPU. The notebook will run on CPU, but the intended target is a **T4 GPU**.

In [ ]:
import os
import platform
import random

import torch

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print(f"Python version: {platform.python_version()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name}")
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print("FP16 inference will be enabled.")
else:
    print("No GPU detected. The notebook will fall back to CPU/FP32 mode.")
    print("For best performance, switch to Runtime > Change runtime type > T4 GPU.")


## 2. Dependency Installation

Install only the packages that are required for this notebook. We intentionally rely on Colab's preinstalled PyTorch runtime and avoid unnecessary extras.

In [ ]:
%pip install -q -U transformers accelerate librosa soundfile safetensors

## 3. Model Loading

We load `openai/whisper-small` with Hugging Face Transformers and build an optimized ASR pipeline.

Key implementation details:

- `torch_dtype=torch.float16` on CUDA to reduce memory pressure and improve throughput.
- `low_cpu_mem_usage=True` for friendlier model initialization in Colab.
- `use_safetensors=True` when available.
- `chunk_length_s=30` for robust long-form transcription.

In [ ]:
from pathlib import Path
from typing import Optional

from IPython.display import Markdown, display
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

MODEL_ID = "openai/whisper-small"
CHUNK_LENGTH_S = 30
BATCH_SIZE = 8
MAX_NEW_TOKENS = 256
TASK = "transcribe"
LANGUAGE: Optional[str] = None  # Example: "en". Keep None for auto language detection.

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
PIPELINE_DEVICE = 0 if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32


def build_transcriber(
    model_id: str,
    device: str,
    pipeline_device,
    torch_dtype: torch.dtype,
    chunk_length_s: int,
    batch_size: int,
    max_new_tokens: int,
):
    """Load Whisper and wrap it in an optimized ASR pipeline."""
    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        use_safetensors=True,
    )
    model.to(device)

    transcriber = pipeline(
        task="automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        chunk_length_s=chunk_length_s,
        batch_size=batch_size,
        max_new_tokens=max_new_tokens,
        return_timestamps=False,
        torch_dtype=torch_dtype,
        device=pipeline_device,
    )
    return transcriber


transcriber = build_transcriber(
    model_id=MODEL_ID,
    device=DEVICE,
    pipeline_device=PIPELINE_DEVICE,
    torch_dtype=TORCH_DTYPE,
    chunk_length_s=CHUNK_LENGTH_S,
    batch_size=BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
)

precision_label = "FP16" if TORCH_DTYPE == torch.float16 else "FP32"
print(f"Loaded model: {MODEL_ID}")
print(f"Device: {DEVICE}")
print(f"Precision: {precision_label}")
print(f"Chunk length: {CHUNK_LENGTH_S}s | Batch size: {BATCH_SIZE}")


## 4. Audio Input Handling

Use Colab's upload widget to send an audio file from your local machine to the runtime. The notebook stores it in `uploaded_audio/` and renders a playback widget so you can verify the uploaded sample before transcription.

In [ ]:
from google.colab import files
from IPython.display import Audio

UPLOAD_DIR = Path("uploaded_audio")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)


def upload_audio_file(upload_dir: Path) -> Path:
    """Upload exactly one audio file into the current Colab runtime."""
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file was uploaded.")

    filename, file_bytes = next(iter(uploaded.items()))
    audio_path = upload_dir / filename
    audio_path.write_bytes(file_bytes)
    return audio_path


audio_path = upload_audio_file(UPLOAD_DIR)
print(f"Uploaded file saved to: {audio_path.resolve()}")
display(Audio(filename=str(audio_path)))


## 5. Inference / Transcription

This cell runs Whisper inference. By default the notebook performs **speech-to-text transcription** in the original spoken language (`task="transcribe"`). If you know the target language ahead of time, set `LANGUAGE` above to reduce ambiguity.

In [ ]:
import json
from time import perf_counter


def transcribe_audio(asr_pipeline, audio_path: Path, task: str = TASK, language: Optional[str] = LANGUAGE):
    """Run Whisper transcription and return text plus run metadata."""
    generate_kwargs = {"task": task}
    if language is not None:
        generate_kwargs["language"] = language

    start_time = perf_counter()
    result = asr_pipeline(str(audio_path), generate_kwargs=generate_kwargs)
    elapsed_time = perf_counter() - start_time

    return {
        "file_name": audio_path.name,
        "file_path": str(audio_path.resolve()),
        "model_id": MODEL_ID,
        "device": DEVICE,
        "precision": "FP16" if TORCH_DTYPE == torch.float16 else "FP32",
        "task": task,
        "language": language if language is not None else "auto",
        "elapsed_time_sec": round(elapsed_time, 3),
        "transcript": result["text"].strip(),
        "raw_result": result,
    }


transcription_result = transcribe_audio(transcriber, audio_path)
print(json.dumps({k: v for k, v in transcription_result.items() if k != 'raw_result'}, indent=2, ensure_ascii=False))


## 6. Output Formatting

Format the transcript for notebook display and persist plain-text plus structured output files. This is useful for reproducibility, downstream automation, or sharing example results in a GitHub repository.

In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def format_transcription_markdown(result: dict) -> str:
    transcript_text = result["transcript"] if result["transcript"] else "_No transcription text returned._"
    return f"""
## Transcription Summary

- **File:** `{result['file_name']}`
- **Model:** `{result['model_id']}`
- **Device:** `{result['device']}`
- **Precision:** `{result['precision']}`
- **Task:** `{result['task']}`
- **Language:** `{result['language']}`
- **Elapsed Time:** `{result['elapsed_time_sec']}` seconds

## Transcript

{transcript_text}
"""


display(Markdown(format_transcription_markdown(transcription_result)))

text_output_path = OUTPUT_DIR / "transcription.txt"
json_output_path = OUTPUT_DIR / "transcription.json"

text_output_path.write_text(transcription_result["transcript"] + "\n", encoding="utf-8")
json_output_path.write_text(
    json.dumps({k: v for k, v in transcription_result.items() if k != "raw_result"}, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print(f"Saved text transcript to: {text_output_path.resolve()}")
print(f"Saved structured metadata to: {json_output_path.resolve()}")

print("\nTip: upload another audio file and re-run the audio input, inference, and output cells to transcribe a new sample.")
